# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-01','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** line item

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd

lines = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])

# Verification assertions
assert len(lines) == 4
assert lines['qty'].sum() == 7

lines

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-01


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
vendors_df = pd.DataFrame(vendors_json)

joined = pd.merge(lines, vendors_df, on='vendor_id', how='left', indicator=True)

unmatched = joined[joined['_merge'] == 'left_only']
print("Unmatched rows:")
print(unmatched)
print(f"\nUnmatched units total: {unmatched['qty'].sum()}")

joined

Unmatched rows:
Empty DataFrame
Columns: [item, qty, order, vendor_id, name, zone, _merge]
Index: []

Unmatched units total: 0


,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-01,Hoos Burgers,A,both


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
# Include dropna=False so unmatched vendors are not silently omitted
vendor_totals = joined.groupby('name', dropna=False)['qty'].sum().reset_index()
print(vendor_totals)

# Comment: We use dropna=False (or group by vendor_id) so that any vendor missing from the vendor reference table still appears in total sales reporting (as NaN) rather than being dropped from business revenue and quantity totals.

           name  qty
0     Cav Merch    1
1  Hoos Burgers    6


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
zone_totals = joined.groupby('zone', dropna=False)['qty'].sum().reset_index()
print(zone_totals)

best_zone = zone_totals.dropna().sort_values(by='qty', ascending=False).iloc[0]
print(f"\nZone selling the most items: Zone {best_zone['zone']} ({best_zone['qty']} units)")

# Note: If an unmatched vendor exists without an assigned zone, its units appear under NaN, ensuring the total across all zones plus NaN equals the full 7 units.

  zone  qty
0    A    7

Zone selling the most items: Zone A (7 units)


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# Ensure every order dictionary has a 'lines' key so json_normalize does not throw a KeyError
ragged_clean = [{**d, 'lines': d.get('lines', [{}])} for d in ragged_json]

ragged_df = pd.json_normalize(ragged_clean, record_path='lines', meta=['order', 'vendor_id'])
ragged_df

,item,qty,order,vendor_id
0,Soda,2.0,4,V-01
1,NaN,NaN,5,V-10
2,Fries,NaN,6,V-01


### Q6 — Validate

In [8]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

**a)** "One row per line item" results in a nice and clean dataset where each row is an atomic transaction of an item. Such a dataset will allow easy computation using pandas' groupby operations by item, vendor, or zone. In contrast, the "one row per order" format confines each item within a nested list in JSON, making it difficult to compute anything at the product level.

**b)** Quantity of 0 is an indication that there has been an order recorded but 0 quantity was delivered (a cancelled line item, for example). Missing quantity (NaN) is an indication of missing information from the API data. The confusion here lies in filling NaNs with zeros (`fillna(0)`), which would affect analytics where `count()` would assume that missing values are actual values and skew averages downward.